# Grocer_Bakery Supplement

The pipeline currently only uses the number of FHRS establishments that are projected to give 95% recall and only aimed to narrow down bakeries using bakery related terms. This method is unlikely to extract in-store bakeries that are hidden within known supermarkets that often do not have a Bakery related front but provide significant provision to the population and are well known for this in the UK.

This notebook aims to supplement these bakeries from the 81,085 FHRS establishments recorded in the initial snapshot.

In [ ]:
from pathlib import Path
import re

import pandas as pd

FHRS_DATA_DATE = "2026-07-23"

RAW_FHRS_PATH = Path(f"../data/business/raw/london_fhrs_raw_{FHRS_DATA_DATE}.csv")

OUTPUT_FOLDER = Path("../data/business/interim/supermarket_bakery")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

CANDIDATES_PATH = OUTPUT_FOLDER / "supermarket_bakery_candidates.csv"
AUTO_INCLUDE_PATH = OUTPUT_FOLDER / "supermarket_bakery_auto_include.csv"
NEEDS_CHECK_PATH = OUTPUT_FOLDER / "supermarket_bakery_needs_check.csv"

fhrs = pd.read_csv(RAW_FHRS_PATH, low_memory=False)

print(f"FHRS establishments: {len(fhrs)}")
print(f"Columns: {len(fhrs.columns)}")

FHRS establishments: 81085
Columns: 28


In [2]:
retail_types = (
    fhrs.loc[fhrs["BusinessType"]
            .str.contains("supermarket|retailer", case=False, na=False),
            "BusinessType"].value_counts())

retail_types

BusinessType
Retailers - other                        16655
Retailers - supermarkets/hypermarkets     2314
Name: count, dtype: int64

In [3]:
CHAIN_PATTERNS = {
    "Tesco": r"\btesco\b",
    "Sainsbury's": (r"\bsainsbury(?:'s|s)?\b"),
    "Asda": r"\basda\b",
    "Morrisons": (r"\bmorrisons\b|\bmorrison's\b|\bw\s*m\s*morrisons?\b"),
    "Lidl": r"\blidl\b",
    "Aldi": r"\baldi\b",
    "Co-op": (r"\bco[- ]?op\b|\bco-operative\b|\bcooperative\b"),
    "Waitrose": r"\bwaitrose\b",
    "M&S": (r"\bmarks\s*(?:&|and)\s*spencer\b|\bm&s\b"),
    "Iceland": r"\biceland\b"
}

In [4]:
def identify_chain(business_name):

    if pd.isna(business_name):
        return None

    business_name = str(business_name)

    for chain, pattern in CHAIN_PATTERNS.items():
        if re.search(pattern, business_name, flags=re.IGNORECASE):
            return chain

    return None


fhrs["Chain"] = (fhrs["BusinessName"].apply(identify_chain))

chain_candidates = (fhrs[fhrs["Chain"].notna()].copy())

print(f"Major-chain FHRS records: {len(chain_candidates)}")

display(chain_candidates["Chain"].value_counts())

Major-chain FHRS records: 1999


Chain
Tesco          564
Sainsbury's    437
Co-op          261
M&S            148
Lidl           128
Waitrose       125
Iceland        120
Asda            85
Morrisons       74
Aldi            57
Name: count, dtype: int64

In [5]:
for chain in sorted(chain_candidates["Chain"]
                    .dropna()
                    .unique()):
    print(f"\n{chain}")

    display(chain_candidates.loc[chain_candidates["Chain"] == chain, "BusinessName"]
            .value_counts()
            .head(3))


Aldi


BusinessName
Aldi                   26
Aldi Stores Limited     9
Aldi Stores Ltd         7
Name: count, dtype: int64


Asda


BusinessName
Asda               25
Asda Express        6
Asda Stores Ltd     3
Name: count, dtype: int64


Co-op


...

BusinessName
Sainsbury's          281
Sainsbury's Local     43
Sainsburys            21
Name: count, dtype: int64


Tesco


BusinessName
Tesco                   315
Tesco Express           125
Tesco Stores Limited     18
Name: count, dtype: int64


Waitrose


BusinessName
Waitrose           65
Little Waitrose     9
Waitrose Ltd        8
Name: count, dtype: int64

In [6]:
# Businesses recorded by the FHRS as supermarkets
supermarket_rows = fhrs[fhrs["BusinessType"] == "Retailers - supermarkets/hypermarkets"].copy()

# Major supermarket chains that FHRS classified as "Retailers - other"
major_chain_other_rows = chain_candidates[chain_candidates["BusinessType"] == "Retailers - other"].copy()

grocer_candidates = pd.concat([supermarket_rows, major_chain_other_rows], ignore_index=True)

# Identify chains again in the combined dataset
grocer_candidates["Chain"] = (grocer_candidates["BusinessName"]
                              .apply(identify_chain))

print(f"Explicit FHRS supermarket records: {len(supermarket_rows)}")
print(f"Additional major-chain retailer records: {len(major_chain_other_rows)}")
print(f"Combined grocery candidates: {len(grocer_candidates)}")

Explicit FHRS supermarket records: 2314
Additional major-chain retailer records: 264
Combined grocery candidates: 2578


In [7]:
grocer_candidates["BusinessNameText"] = (grocer_candidates["BusinessName"]
                                         .fillna("")
                                         .astype(str)
                                         .str.lower()
                                         .str.strip())

ADDRESS_COLUMNS = [
    "AddressLine1",
    "AddressLine2",
    "AddressLine3",
    "PostCode"]


def combine_address(row):
    address_parts = []

    for value in row:
        if pd.notna(value):
            value = str(value).strip()
            if value:
                address_parts.append(value)

    return ", ".join(address_parts)


grocer_candidates["Address"] = (grocer_candidates[ADDRESS_COLUMNS]
                                .apply(combine_address, axis=1))

In [8]:
CHAIN_START_PATTERNS = {
    "Tesco": r"^tesco\b",
    "Sainsbury's": r"^(?:j\s+)?sainsbury(?:'s|s)?\b",
    "Asda": r"^asda\b",
    "Morrisons": r"^(?:morrisons?|w\s*m\s*morrisons?)\b",
    "Lidl": r"^lidl\b",
    "Aldi": r"^aldi\b",
    "Co-op": r"^(?:the\s+)?(?:co[- ]?op|co-operative|cooperative)\b",
    "Waitrose": r"^waitrose\b",
    "M&S": r"^(?:marks\s*(?:&|and)\s*spencer|m&s)\b",
    "Iceland": r"^iceland\b"}


def screen_grocer(row):
    chain = row["Chain"]
    name = row["BusinessNameText"]

    if pd.notna(chain):
        chain_pattern = CHAIN_START_PATTERNS[chain]

        if not re.search(chain_pattern, name):
            return ("AUTO_EXCLUDE", "Separate business mentioning supermarket chain")

    # Tesco
    if chain == "Tesco":
        if re.search(r"\bexpress\b|\bgarage\b|\bpetrol\b|\btesco\.com\b", name):
            return ("AUTO_EXCLUDE", "Tesco convenience/non-supermarket format")

        if re.search(r"\bextra\b|\bsuperstore\b|\bsupermarket\b|\bmain store\b", name):
            return ("AUTO_INCLUDE", "Large Tesco supermarket format")

        return ("NEEDS_CHECK", "Tesco store format unclear from FHRS record")


    # Sainsbury's
    if chain == "Sainsbury's":
        if re.search(r"\blocal\b|\bpetrol\b|\bfilling station\b", name):
            return ("AUTO_EXCLUDE", "Sainsbury's convenience/non-supermarket format")

        if re.search(r"\bsupermarket\b", name):
            return ("AUTO_INCLUDE", "Large Sainsbury's supermarket format")

        return ("NEEDS_CHECK", "Sainsbury's store format unclear from FHRS record")

    # Asda
    if chain == "Asda":
        if re.search(r"\bexpress\b|\bpetrol\b|\bfilling station\b", name):
            return ("AUTO_EXCLUDE", "Asda convenience/non-supermarket format")

        if re.search(r"\bsuperstore\b|\bsupermarket\b", name):
            return ("AUTO_INCLUDE", "Large Asda supermarket format")

        return ("NEEDS_CHECK", "Asda store format unclear from FHRS record")

    # Morrisons
    if chain == "Morrisons":
        if re.search(r"\bdaily\b|\bm local\b|\bpetrol\b|\bgarage\b", name):
            return ("AUTO_EXCLUDE", "Morrisons convenience/non-supermarket format")

        return ("AUTO_INCLUDE", "Full-format Morrisons supermarket")

    # Lidl
    if chain == "Lidl":
        return ("AUTO_INCLUDE", "Lidl supermarket with established bakery provision")


    # Co-op
    if chain == "Co-op":
        return ("AUTO_EXCLUDE", "Convenience/local grocery format excluded from supplement")


    # Aldi
    if chain == "Aldi":
        return ("AUTO_EXCLUDE", "No qualifying in-store bakery provision assumed")

    # Iceland
    if chain == "Iceland":
        return ("AUTO_EXCLUDE", "No qualifying in-store bakery provision assumed")

    # --------------------------------------------------
    # Waitrose
    # --------------------------------------------------

    if chain == "Waitrose":
        if re.search(r"\blittle waitrose\b|\bshell\b|\brecharge\b", name):
            return ("AUTO_EXCLUDE", "Waitrose convenience/concession format")

        return ("NEEDS_CHECK", "Full Waitrose store: bakery provision to confirm")


    # M&S
    if chain == "M&S":
        if re.search(r"\bsimply food\b", name):
            return ("AUTO_EXCLUDE", "M&S convenience food format")

        return ("NEEDS_CHECK", "Larger M&S store: bakery provision to confirm")

    # Other / independent supermarket
    return ("NEEDS_CHECK", "Independent or other supermarket")


grocer_candidates[
    ["ScreenStatus", "ScreenReason"]] = grocer_candidates.apply(screen_grocer, axis=1, result_type="expand")

In [9]:
print("Screening results:")
display(grocer_candidates["ScreenStatus"].value_counts())

print("\nScreening results by chain:")
screening_by_chain = pd.crosstab(
    grocer_candidates["Chain"].fillna("Other / independent"),
    grocer_candidates["ScreenStatus"])
display(screening_by_chain)

Screening results:


ScreenStatus
NEEDS_CHECK     1609
AUTO_EXCLUDE     785
AUTO_INCLUDE     184
Name: count, dtype: int64


Screening results by chain:


ScreenStatus,AUTO_EXCLUDE,AUTO_INCLUDE,NEEDS_CHECK
Chain,,,
Aldi,57,0,0
Asda,33,3,37
Co-op,246,0,0
Iceland,119,0,0
Lidl,0,126,0
M&S,47,0,88
Morrisons,44,25,0
Other / independent,0,0,692
Sainsbury's,63,14,347


In [10]:
auto_include = grocer_candidates[grocer_candidates["ScreenStatus"] == "AUTO_INCLUDE"].copy()

auto_exclude = grocer_candidates[grocer_candidates["ScreenStatus"] == "AUTO_EXCLUDE"].copy()

needs_check = grocer_candidates[grocer_candidates["ScreenStatus"] == "NEEDS_CHECK"].copy()


print(f"Automatically included: {len(auto_include)}")
print(f"Automatically excluded: {len(auto_exclude)}")
print(f"Needs checking: {len(needs_check)}")

Automatically included: 184
Automatically excluded: 785
Needs checking: 1609


# Checking for repeated counts of stores

Many supermarkets and grocers have multiple subsidiary brands that may also have a record in the FHRS, this can be removed and preven duplication in checks.

In [11]:
grocer_candidates["PostCodeClean"] = (grocer_candidates["PostCode"]
                                      .fillna("")
                                      .astype(str)
                                      .str.upper()
                                      .str.replace(" ", "", regex=False))

In [12]:
site_counts = (grocer_candidates[
    grocer_candidates["Chain"].notna() & grocer_candidates["PostCodeClean"].ne("")]
    .groupby(["Chain", "PostCodeClean"])
    .size()
    .reset_index(name="SiteRecords"))

In [13]:
grocer_candidates = grocer_candidates.merge(site_counts,
                                            on=["Chain", "PostCodeClean"],
                                            how="left")

grocer_candidates["SiteRecords"] = (grocer_candidates["SiteRecords"]
                                    .fillna(1)
                                    .astype(int))

In [14]:
print(f"Chain locations sharing a postcode: {(grocer_candidates["SiteRecords"] > 1).sum()}")

Chain locations sharing a postcode: 115


In [15]:
display(grocer_candidates[grocer_candidates["SiteRecords"] > 1][
    ["Chain",
     "BusinessName",
     "Address",
     "PostCode",
     "ScreenStatus"]]
    .sort_values(["Chain", "PostCode"])
    .head(10))

,Chain,BusinessName,Address,PostCode,ScreenStatus
2122,Asda,Asda,"Asda Superstore 1 Leyton Mills Marshall Road, ...",E10 5NH,NEEDS_CHECK
2545,Asda,Haji Baba Halal Meat at Asda,"Arc Car Wash Leyton Mills Marshall Road, Leyto...",E10 5NH,AUTO_EXCLUDE
2500,Asda,Euro Garages Ltd (Asda Express),"Aldersbrook Service Station, 19 Aldersbrook Ro...",E12 5HH,AUTO_EXCLUDE
2501,Asda,Asda Express Greggs Aldersbrook,"Aldersbrook Service Station, 19 Aldersbrook Ro...",E12 5HH,AUTO_EXCLUDE
2135,Asda,Asda Supermarket,"19 Selborne Walk, Walthamstow, E17 7JR",E17 7JR,AUTO_INCLUDE
2546,Asda,Haji Baba Halal Meat at Asda,"19 Selborne Walk, Walthamstow, E17 7JR",E17 7JR,AUTO_EXCLUDE
848,Asda,Asda,"130 Chase Side, LONDON, N14 5PW",N14 5PW,NEEDS_CHECK
2375,Asda,"Haji Baba Ltd, Asda Superstore","130 Chase Side, LONDON, N14 5PW",N14 5PW,AUTO_EXCLUDE
873,Asda,Asda,"1 West Mall, LONDON, N9 0AL",N9 0AL,NEEDS_CHECK
2373,Asda,Haji Baba Ltd c/o Asda,"1 West Mall, LONDON, N9 0AL",N9 0AL,AUTO_EXCLUDE


In [16]:
grocer_candidates.to_csv(CANDIDATES_PATH, index=False)
auto_include.to_csv(AUTO_INCLUDE_PATH, index=False)
needs_check.to_csv(NEEDS_CHECK_PATH, index=False)

print(f"All candidates: {len(grocer_candidates)}")
print(f"Automatically included: {len(auto_include)}")
print(f"Automatically excluded: {len(auto_exclude)}")
print(f"Needs AI checking: {len(needs_check)}")

All candidates: 2578
Automatically included: 184
Automatically excluded: 785
Needs AI checking: 1609


# AI verification for remaining supermarkets

After screening the grocery establishments, some remain where the store format or level of bakery provision remain difficult to establish from FHRS records alone. The establishments under needs_checks as well as the auto_includes will be AI verified at this stage (auto_includes seem to not always be the type of store that the FHRS record makes them out to be by name). This will ensure that the identity can be better understood and accurate inclusions can be made to our Grocer_Bakery set.

In [17]:
auto_include["InitialDecision"] = "AUTO_INCLUDE"
needs_check["InitialDecision"] = "NEEDS_CHECK"

ai_candidates = pd.concat([auto_include, needs_check], ignore_index=True)

print(f"AI verification candidates: {len(ai_candidates)}")

display(ai_candidates["InitialDecision"].value_counts())

AI verification candidates: 1793


InitialDecision
NEEDS_CHECK     1609
AUTO_INCLUDE     184
Name: count, dtype: int64

In [ ]:
from getpass import getpass
from datetime import datetime

import time

from google import genai
from google.genai import types, errors

MODEL = "gemini-2.5-flash"
RUN_API = False
RUN_TEST_BATCH = False

BATCH_SIZE = 10
REQUEST_DELAY = 2

AI_RESULTS_PATH = (OUTPUT_FOLDER / "supermarket_bakery_ai_results.csv")

In [19]:
client = None

In [20]:
if AI_RESULTS_PATH.exists():
    previous_results = pd.read_csv(AI_RESULTS_PATH)
    completed_ids = set(previous_results["FHRSID"].astype(str))

else:
    completed_ids = set()

remaining_queue = ai_candidates[~ai_candidates["FHRSID"]
                                .astype(str)
                                .isin(completed_ids)].copy()

remaining_queue = remaining_queue.reset_index(drop=True)

requests_needed = (len(remaining_queue) + BATCH_SIZE - 1) // BATCH_SIZE

if RUN_API and len(remaining_queue) > 0:
    api_key = getpass("Gemini API key: ")
    client = genai.Client(api_key=api_key)
    
    print(f"Gemini client ready for {len(remaining_queue)} remaining candidates.")

elif len(remaining_queue) == 0:
    print("No Gemini API requests required. Existing supermarket verification results are complete.")

else:
    print(f"Gemini API calls disabled. {len(remaining_queue)} candidates remain in the queue.")

print(f"Total candidates: {len(ai_candidates)}")
print(f"Already completed: {len(completed_ids)}")
print(f"Remaining: {len(remaining_queue)}")
print(f"Maximum requests: {requests_needed}")

No Gemini API requests required. Existing supermarket verification results are complete.
Total candidates: 1793
Already completed: 1791
Remaining: 0
Maximum requests: 0


In [21]:
def clean_value(value):
    if pd.isna(value):
        return "Unknown"
    return str(value)


def build_grocer_prompt(batch):
    batch = batch.reset_index(drop=True)

    businesses = []
    for i, row in batch.iterrows():
        businesses.append(
            f"""
BUSINESS {i + 1}
Name: {clean_value(row["BusinessName"])}
Chain: {clean_value(row["Chain"])}
Address: {clean_value(row["Address"])}
Postcode: {clean_value(row["PostCode"])}
Local authority: {clean_value(row["LocalAuthorityName"])}
Initial screening: {clean_value(row["InitialDecision"])}
""".strip())

    business_text = "\n\n".join(businesses)

    prompt = f"""

ROLE
    
You are verifying supermarket bakery provision in London.

Use Google Search to investigate each of the {len(batch)} specific grocery locations below.

For each business:

1. Identify the specific store using its name, address and postcode.
2. Decide whether that location has substantial customer-facing bakery provision.

The initial screening result is supplied only for context.
Independently verify the location and do not assume that it is correct.

BAKERY PROVISION = YES

Use YES where there is reasonable evidence of substantial bakery provision, such as:
- an in-store bakery;
- a bakery counter or department;
- a substantial fresh bread and pastry section;
- a substantial range of bakery products baked or finished in store.

Products do not have to be made from raw ingredients on site.
Bake-off bakery provision can qualify.

BAKERY PROVISION = NO

Use NO where:
- the store only sells ordinary packaged bakery products;
- bakery products are incidental;
- it is a small convenience format without substantial bakery provision;
- it is a petrol station or similar small-format store;
- it is actually a separate concession or other business rather than
  the supermarket.

Examples of formats that should normally be NO unless strong evidence shows substantial bakery provision include Tesco Express, Sainsbury's Local, 
Morrisons Daily, Asda Express, Little Waitrose and M&S Simply Food.

UNCLEAR

Use UNCLEAR if the exact branch cannot reasonably be identified or there is insufficient evidence about bakery provision.

Do not assume every branch of a chain has the same facilities.

Prefer official supermarket/store-locator evidence where available.

OUTPUT

Return exactly {len(batch)} lines:

business number | bakery provision | short reason

Bakery provision must be:
YES, NO or UNCLEAR

Example:

1 | YES | Official store page lists an in-store bakery.
2 | NO | Exact location is a Tesco Express convenience store.
3 | UNCLEAR | Exact branch could not be confidently identified.

Return every business number exactly once.
Keep each reason to one short sentence.
Return only the result lines.


{business_text}
""".strip()

    return prompt

In [22]:
def verify_batch(batch):
    prompt = build_grocer_prompt(batch)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0,
            max_output_tokens=800,
            thinking_config=types.ThinkingConfig(thinking_budget=0)))

    return response

In [23]:
def parse_response(response_text, batch):

    batch = batch.reset_index(drop=True)
    text = response_text.strip()

    # Helps if Gemini accidentally joins result lines
    for number in range(len(batch), 0, -1):
        text = text.replace(f"{number} |", f"\n{number} |")

    parsed = {}

    for line in text.splitlines():

        parts = [part.strip()
                 for part in line.split("|", 2)]

        if len(parts) != 3:
            continue

        number_text = (parts[0]
                       .replace("*", "")
                       .strip())

        if not number_text.isdigit():
            continue

        number = int(number_text)

        if not 1 <= number <= len(batch):
            continue

        decision = parts[1].upper()

        if decision not in {"YES", "NO", "UNCLEAR"}:
            continue

        row = batch.iloc[number - 1]

        parsed[number] = {
            "FHRSID": row["FHRSID"],
            "BusinessName": row["BusinessName"],
            "Chain": row["Chain"],
            "BusinessType": row["BusinessType"],
            "Address": row["Address"],
            "PostCode": row["PostCode"],
            "LocalAuthorityName": row["LocalAuthorityName"],
            "InitialDecision": row["InitialDecision"],
            "BakeryProvision": decision,
            "AIReason": parts[2],
            "Model": MODEL,
            "VerificationDateTime": datetime.now().isoformat(timespec="seconds")}

    results = pd.DataFrame([parsed[number]
                            for number in sorted(parsed)])

    return results

In [24]:
def save_results(results):

    if results.empty:
        return

    results.to_csv(AI_RESULTS_PATH, mode="a", header=not AI_RESULTS_PATH.exists(), index=False)

In [ ]:
test_results = pd.DataFrame()

if RUN_API and RUN_TEST_BATCH and len(remaining_queue) > 0:

    test_batch = (remaining_queue.head(10)
                  .copy()
                  .reset_index(drop=True))

    test_response = verify_batch(test_batch)
    test_results = parse_response(test_response.text, test_batch)

    display(test_results[["BusinessName",
                          "PostCode",
                          "InitialDecision",
                          "BakeryProvision",
                          "AIReason"]])

    print(f"Parsed {len(test_results)} out of {len(test_batch)}")

    save_results(test_results)

    # Remove successfully tested businesses from production queue
    tested_ids = set(test_results["FHRSID"].astype(str))

    remaining_queue = (remaining_queue[~remaining_queue["FHRSID"]
                                       .astype(str)
                                       .isin(tested_ids)].copy().reset_index(drop=True))

    print(f"Saved {len(test_results)} test results.")
    print(f"Remaining after test batch: {len(remaining_queue)}")

else:
    print("Test API batch skipped.")

Test API batch skipped.


In [26]:
requests_attempted = 0
results_saved = 0

if RUN_API:

    for start in range(0, len(remaining_queue), BATCH_SIZE):

        batch = (remaining_queue.iloc[start: start + BATCH_SIZE]
                .copy()
                .reset_index(drop=True))

        requests_attempted += 1

        try:
            response = verify_batch(batch)
            results = parse_response(response.text, batch)
            save_results(results)

            results_saved += len(results)

            print(f"Batch {requests_attempted}: saved {len(results)}/{len(batch)}")

        except errors.APIError as error:
            print(f"Batch {requests_attempted} failed: {error.code}")

            if error.code == 429:
                print("Rate limit reached. Stopping safely.")
                break

        except Exception as error:
            print(f"Batch {requests_attempted} failed: {error}")

        time.sleep(REQUEST_DELAY)

else:
    print("No batches in queue; skipping Gemini verification. Existing saved results will be used.")

No batches in queue; skipping Gemini verification. Existing saved results will be used.


In [27]:
ai_results = pd.read_csv(AI_RESULTS_PATH)

print(f"Completed: {len(ai_results)} / {len(ai_candidates)}")

print("\nAI decisions:")
display(ai_results["BakeryProvision"].value_counts())

Completed: 1793 / 1793

AI decisions:


BakeryProvision
NO         758
YES        708
UNCLEAR    327
Name: count, dtype: int64

In [28]:
ai_results = ai_results.drop_duplicates(subset="FHRSID", keep="first").copy()

print(f"Unique AI results: {len(ai_results)}")

Unique AI results: 1791


In [29]:
unclear_review = ai_results[ai_results["BakeryProvision"] == "UNCLEAR"].copy()

unclear_review = unclear_review.drop_duplicates(subset="FHRSID", keep="first")

unclear_review = unclear_review[
    ["FHRSID",
     "BusinessName",
     "Chain",
     "Address",
     "PostCode",
     "InitialDecision",
     "AIReason"]].copy()

unclear_review["ManualDecision"] = ""
unclear_review["ManualReason"] = ""

print(f"Unclear locations to review: {len(unclear_review)}")
display(unclear_review.head(20))

Unclear locations to review: 327


,FHRSID,BusinessName,Chain,Address,PostCode,InitialDecision,AIReason,ManualDecision,ManualReason
19,858154,Morrisons,Morrisons,"399 Edgware Road, Kingsbury, London, NW9 0FH",NW9 0FH,AUTO_INCLUDE,Insufficient evidence to confirm substantial b...,,
24,940538,Lidl,Lidl,"2A Glebe Way, West Wickham, BR4 0RJ",BR4 0RJ,AUTO_INCLUDE,Insufficient information to confirm substantia...,,
25,1094617,Lidl,Lidl,"Ground Floor 124 - 126 High Street, Bromley, B...",BR1 1EZ,AUTO_INCLUDE,Insufficient information to confirm substantia...,,
27,1450625,Lidl,Lidl,"Unit 1 168 Sevenoaks Way, Orpington, BR5 3AE",BR5 3AE,AUTO_INCLUDE,Insufficient information to confirm substantia...,,
33,470814,Sainsburys Supermarket,Sainsbury's,"204-206 West End Lane, NW6 1SG",NW6 1SG,AUTO_INCLUDE,The Sainsbury's store locator for 204-206 West...,,
51,1108943,Lidl,Lidl,"Unit 4, The Oaks Shopping Centre, High Street,...",W3 6RE,AUTO_INCLUDE,Insufficient information to confirm substantia...,,
52,1108944,Lidl,Lidl,"63 - 65 Broadway, West Ealing, W13 9BP",W13 9BP,AUTO_INCLUDE,Insufficient information to confirm substantia...,,
55,1582816,Lidl,Lidl,"23 Old Oak Common Lane, Acton, W3 7BA",W3 7BA,AUTO_INCLUDE,Insufficient information to confirm substantia...,,
58,316031,Wm Morrison Supermarkets Limited,Morrisons,"Morrisons Supermarket And Car Park, Aldermans ...",N13 4YD,AUTO_INCLUDE,Insufficient information to confirm substantia...,,
64,1030259,Lidl,Lidl,"654-658 Hertford Road, ENFIELD, EN3 6LZ",EN3 6LZ,AUTO_INCLUDE,"While Lidl Hertford Road is a supermarket, spe...",,


In [30]:
MANUAL_REVIEW_PATH = (OUTPUT_FOLDER / "supermarket_bakery_manual_review.csv")

if MANUAL_REVIEW_PATH.exists():
    print("The existing manual-review file has not been overwritten.")
else:
    unclear_review.to_csv(MANUAL_REVIEW_PATH, index=False)
    print(f"Saved to: {MANUAL_REVIEW_PATH}")

The existing manual-review file has not been overwritten.


# After manual-review

Collating the manual review outcomes and creating one final csv for supplemental bakeries.

In [31]:
ai_results = pd.read_csv(AI_RESULTS_PATH)
manual_review = pd.read_csv(MANUAL_REVIEW_PATH)

# Remove the two duplicated AI result rows
ai_results = ai_results.drop_duplicates(subset="FHRSID", keep="first").copy()

# Standardise the manual decisions
manual_review["ManualDecision"] = (manual_review["ManualDecision"]
                                   .fillna("")
                                   .str.upper()
                                   .str.strip())

print(f"Unique AI results: {len(ai_results)}")
print(f"Manual reviews: {len(manual_review)}")

Unique AI results: 1791
Manual reviews: 327


In [32]:
manual_decisions = manual_review[
    ["FHRSID",
     "ManualDecision",
     "ManualReason"]].copy()

final_decisions = ai_results.merge(manual_decisions, on="FHRSID", how="left")

In [33]:
final_decisions["FinalBakeryProvision"] = (final_decisions["BakeryProvision"])

post_manual_review = (final_decisions["ManualDecision"].notna()
                      & final_decisions["ManualDecision"].ne(""))

final_decisions.loc[post_manual_review, "FinalBakeryProvision"] = final_decisions.loc[post_manual_review, "ManualDecision"]

In [34]:
display(final_decisions["FinalBakeryProvision"].value_counts())

print(f"\nRemaining UNCLEAR: {(final_decisions['FinalBakeryProvision'] == 'UNCLEAR').sum()}")

FinalBakeryProvision
NO     1029
YES     762
Name: count, dtype: int64


Remaining UNCLEAR: 0


In [35]:
yes_decisions = final_decisions[final_decisions["FinalBakeryProvision"] == "YES"].copy()

grocer_bakeries = grocer_candidates.merge(
    yes_decisions[
        ["FHRSID",
         "FinalBakeryProvision",
         "AIReason",
         "ManualDecision",
         "ManualReason"]],
         on="FHRSID",
         how="inner")

grocer_bakeries = grocer_bakeries.drop_duplicates(subset="FHRSID", keep="first").copy()

print(f"Qualifying records: {len(grocer_bakeries)}")

Qualifying records: 762


In [36]:
grocer_bakeries["PostCodeClean"] = (grocer_bakeries["PostCode"]
                                    .fillna("")
                                    .str.upper()
                                    .str.replace(" ", "", regex=False))

chain_bakeries = grocer_bakeries[grocer_bakeries["Chain"].notna()].copy()
other_bakeries = grocer_bakeries[grocer_bakeries["Chain"].isna()].copy()

chain_bakeries = chain_bakeries.drop_duplicates(subset=["Chain", "PostCodeClean"], keep="first")
grocer_bakeries_final = pd.concat([chain_bakeries, other_bakeries], ignore_index=True)

print(f"Before site deduplication: {len(grocer_bakeries)}")
print(f"Final locations: {len(grocer_bakeries_final)}")

Before site deduplication: 762
Final locations: 749


In [37]:
grocer_bakeries_final["BakeryClass"] = "GROCER_BAKERY"

FINAL_GROCER_PATH = (OUTPUT_FOLDER / "supermarket_bakery_final.csv")

grocer_bakeries_final.to_csv(FINAL_GROCER_PATH, index=False)

print(f"Saved {len(grocer_bakeries_final)} GROCER_BAKERY locations.")
print(FINAL_GROCER_PATH)

Saved 749 GROCER_BAKERY locations.
..\data\business\interim\supermarket_bakery\supermarket_bakery_final.csv
